In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import h5py
import pickle 
import numpy as np 
import pyBigWig
from helpers import * 
from kerasAC.helpers.format_interpretations import * 
from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr, spearmanr

/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:172: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  min_coords = np.vstack(data.min(0) for data in polygons_data).min(0)
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:173: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  max_coords = np.vstack(data.max(0) for data in polygons_data).max(0)


In [2]:
cell_line="K562"
fold=0

In [3]:
vierstra=pyBigWig.open('/srv/scratch/annashch/chrombpnet/accuracy_footprint_compare/vierstra/K562.DS16924.interval.all.fpr.bw','r')

In [4]:
tobias=pyBigWig.open("/oak/stanford/groups/akundaje/projects/enzymatic_bias_correction/tobias/dnase/footprints/K562_idr_footprints.bw",'r')

In [5]:
cur_shap=pickle.load(open("/srv/scratch/annashch/chrombpnet/k562_dnase/bpnet/interpret/K562.10k.DNASE.bias_corrected_bpnet_tobias.fold"+str(fold)+".deepSHAP",'rb'))


In [6]:
cur_shap.keys()

dict_keys(['label_prof', 'label_sum', 'pred_prof', 'pred_sum', 'profile_shap', 'count_shap', 'seq'])

In [8]:
class_shap=format_binary_deepshap("/srv/scratch/annashch/bias_correction/uncorrected/dnase/classification/K562/interpretation/10k.DNASE.K562."+str(fold)+".deepSHAP.interp.npz")    


In [9]:
reg_shap=format_binary_deepshap("/srv/scratch/annashch/bias_correction/uncorrected/dnase/regression/K562/interpretation/10k.DNASE.K562."+str(fold)+".deepSHAP.interp.npz")


In [12]:
gkm_scores=format_gkm_scores("/oak/stanford/groups/akundaje/projects/enzymatic_bias_correction/svm/dnase/K562/interpret/gkmexplain.K562."+str(fold)+".txt")


In [14]:
coords=list(gkm_scores.keys())


In [16]:
coords[0:10]

[('chr16', 68988100),
 ('chr14', 81009527),
 ('chr4', 184461106),
 ('chr7', 38374640),
 ('chr19', 13936311),
 ('chr20', 63864353),
 ('chr1', 28927947),
 ('chr13', 80341396),
 ('chr9', 105694502),
 ('chr10', 101575773)]

In [17]:
cor_dict={} 
cor_dict['tobias_spearman']={}
cor_dict['tobias_pearson']={}
cor_dict['vierstra_spearman']={}
cor_dict['vierstra_pearson']={}
cor_dict['tobias_spearman_random']={}
cor_dict['tobias_pearson_random']={} 
cor_dict['vierstra_spearman_random']={}
cor_dict['vierstra_pearson_random']={}

for key in cor_dict: 
    cor_dict[key]['profile']=[]
    cor_dict[key]['counts']=[]
    cor_dict[key]['class']=[]
    cor_dict[key]['reg']=[]
    cor_dict[key]['gkm']=[]
    

i=0
for coord in coords: 
    i+=1
    if i%100==0: 
        print(str(i))
    #get the tobias footprint scores 
    tobias_vals=np.nan_to_num(tobias.values(coord[0],coord[1]-500,coord[1]+500))
    
    #get the vierstra footprint scores 
    vierstra_vals=1-np.nan_to_num(vierstra.values(coord[0],coord[1]-500,coord[1]+500))
    
    
    #print(np.sum(tobias_vals-vierstra_vals))
    #get the bpnet profile & count shap values 
    label_prob, label_sum, pred_prob, pred_sum, profile_shap, count_shap, seq, minval_perf, maxval_perf, minval_shap, maxval_shap=extract_region(coord,cur_shap)
    try:
        #classification shap 
        binary_class_shap=class_shap[coord]
    except:
        continue 
    #regression shap 
    binary_reg_shap=reg_shap[coord]

    #gkm explain score 
    gkm_score=gkm_scores[coord]
    
    #get observed deepSHAP scores 
    profile_shap_observed=smooth(project_scores(profile_shap,seq))
    count_shap_observed=smooth(project_scores(count_shap,seq))
    class_shap_observed=smooth(project_scores(binary_class_shap,seq))
    reg_shap_observed=smooth(project_scores(binary_reg_shap,seq))
    gkm_observed=smooth(project_scores(gkm_score,seq))
    
    #get pearson & spearman 
    cor_dict['tobias_spearman']['profile'].append(spearmanr(profile_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson']['profile'].append(pearsonr(profile_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman']['profile'].append(spearmanr(profile_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson']['profile'].append(pearsonr(profile_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman']['counts'].append(spearmanr(count_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson']['counts'].append(pearsonr(count_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman']['counts'].append(spearmanr(count_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson']['counts'].append(pearsonr(count_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman']['class'].append(spearmanr(class_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson']['class'].append(pearsonr(class_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman']['class'].append(spearmanr(class_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson']['class'].append(pearsonr(class_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman']['reg'].append(spearmanr(reg_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson']['reg'].append(pearsonr(reg_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman']['reg'].append(spearmanr(reg_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson']['reg'].append(pearsonr(reg_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman']['gkm'].append(spearmanr(gkm_observed,tobias_vals)[0])
    cor_dict['tobias_pearson']['gkm'].append(pearsonr(gkm_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman']['gkm'].append(spearmanr(gkm_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson']['gkm'].append(pearsonr(gkm_observed,vierstra_vals)[0])
    
    #randomize tobias profiles 
    np.random.shuffle(tobias_vals)
    #randomize vierstra profiles
    np.random.shuffle(vierstra_vals)
    
    #get pearson & spearman against the randomly shuffled tobias/vierstra profiles
    cor_dict['tobias_spearman_random']['profile'].append(spearmanr(profile_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson_random']['profile'].append(pearsonr(profile_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman_random']['profile'].append(spearmanr(profile_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson_random']['profile'].append(pearsonr(profile_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman_random']['counts'].append(spearmanr(count_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson_random']['counts'].append(pearsonr(count_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman_random']['counts'].append(spearmanr(count_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson_random']['counts'].append(pearsonr(count_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman_random']['class'].append(spearmanr(class_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson_random']['class'].append(pearsonr(class_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman_random']['class'].append(spearmanr(class_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson_random']['class'].append(pearsonr(class_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman_random']['reg'].append(spearmanr(reg_shap_observed,tobias_vals)[0])
    cor_dict['tobias_pearson_random']['reg'].append(pearsonr(reg_shap_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman_random']['reg'].append(spearmanr(reg_shap_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson_random']['reg'].append(pearsonr(reg_shap_observed,vierstra_vals)[0])
    
    
    cor_dict['tobias_spearman_random']['gkm'].append(spearmanr(gkm_observed,tobias_vals)[0])
    cor_dict['tobias_pearson_random']['gkm'].append(pearsonr(gkm_observed,tobias_vals)[0])
    cor_dict['vierstra_spearman_random']['gkm'].append(spearmanr(gkm_observed,vierstra_vals)[0])
    cor_dict['vierstra_pearson_random']['gkm'].append(pearsonr(gkm_observed,vierstra_vals)[0])


100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000


In [27]:
tb_spear_df=pd.DataFrame.from_dict(cor_dict['tobias_spearman'])
tb_spear_df['cortype']='spearman'
tb_spear_df['motifs']='tobias'

vr_spear_df=pd.DataFrame.from_dict(cor_dict['vierstra_spearman'])
vr_spear_df['cortype']='spearman'
vr_spear_df['motifs']='vierstra'

tb_pearson_df=pd.DataFrame.from_dict(cor_dict['tobias_pearson'])
tb_pearson_df['cortype']='pearson'
tb_pearson_df['motifs']='tobias'

vr_pearson_df=pd.DataFrame.from_dict(cor_dict['vierstra_pearson'])
vr_pearson_df['cortype']='pearson'
vr_pearson_df['motifs']='vierstra'


tb_spear_df_shuff=pd.DataFrame.from_dict(cor_dict['tobias_spearman_random'])
tb_spear_df_shuff['cortype']='spearman_shuff'
tb_spear_df_shuff['motifs']='tobias'

vr_spear_df_shuff=pd.DataFrame.from_dict(cor_dict['vierstra_spearman_random'])
vr_spear_df_shuff['cortype']='spearman_shuff'
vr_spear_df_shuff['motifs']='vierstra'

tb_pearson_df_shuff=pd.DataFrame.from_dict(cor_dict['tobias_pearson_random'])
tb_pearson_df_shuff['cortype']='pearson_shuff'
tb_pearson_df_shuff['motifs']='tobias'

vr_pearson_df_shuff=pd.DataFrame.from_dict(cor_dict['vierstra_pearson_random'])
vr_pearson_df_shuff['cortype']='pearson_shuff'
vr_pearson_df_shuff['motifs']='vierstra'



#concatenate the dataframes
all_df=pd.concat([tb_spear_df,vr_spear_df,tb_pearson_df,vr_pearson_df,tb_spear_df_shuff,vr_spear_df_shuff,tb_pearson_df_shuff,vr_pearson_df_shuff],axis=0)
#export to csv for visualization in R (better ggridges package )
all_df.to_csv("Tobias_Vierstra_Cor.K562.DNASE.10k.Smoothed.Inverted.WithShuffle.txt",header=True,index=False,sep='\t')

In [28]:
all_df.tail()

,profile,counts,class,reg,gkm,cortype,motifs
9994,-0.030226,-0.052735,0.021060,-0.026746,0.053653,pearson_shuff,vierstra
9995,0.015974,0.012929,0.038013,0.022808,0.025219,pearson_shuff,vierstra
9996,0.007150,0.004157,-0.002103,-0.011505,-0.013636,pearson_shuff,vierstra
9997,0.038872,0.007242,0.059162,0.062720,0.068071,pearson_shuff,vierstra
9998,0.045428,0.030802,0.029179,0.015366,0.016129,pearson_shuff,vierstra
